# 1. Carregamento de dados All Pre Alocated Ids

In [24]:
import functions

pre_aloc = functions.carregamento(
path='C:/Users/luiz.farias/Numerator International/BKO - projeto-dados-ops/do_preal_fornecedor/ref_prealoc',
prefixo='pre'
)

trocas = {
'{GroupId}' : 'UserPS',
'{IndividualId}' : 'individual-id',
'{CreationTimeStamp}' : 'created_at',
'Origen' : 'Origen_GPM'
}

pre_aloc.rename(columns=trocas, inplace=True)
pre_aloc.UserPS.nunique()


Cons = functions.carregamento(
path='C:/Users/luiz.farias/Numerator International/BKO - projeto-dados-ops/do_preal_fornecedor/ref_cosolidado',
prefixo='consolidado'
)
pre_aloc.UserPS.nunique()

KeyboardInterrupt: 

In [ ]:
import functions

troca = functions.carregamento(
path='C:/Users/luiz.farias/Numerator International/BKO - projeto-dados-ops/do_preal_fornecedor/ref_troca',
prefixo='ref_'
)

trocas = {
'{GroupId}' : 'UserPS',
'{IndividualId}' : 'individual-id',
'{CreationTimeStamp}' : 'created_at',
'Origen' : 'Origen_GPM'
}

troca.rename(columns=trocas, inplace=True)
troca.UserPS.nunique()

7698

# 2. Pesquisa

In [ ]:
# Incluír ids ou lista de ids separados por vírgula ex: [552229497, 552229823, 552221243]
src = [552190469,552155252,552189962,552187380,552190054,552155350,552155388,552158023,552154840,552189760,552187752,552187293,552190603,552157702,552154844,552154632,552154693,
       552154548,552155217,552154531,552190050,552155143,552190395,552160661,552155386,552160828,552155415,552155383,552154537,552160594,552154707,552154666,552154759,552190324,
       552154673,552155216,552190093,552160598,552189915,552185366,552189991,552190278,552190221,552157692,552157839,552187345,552185292,552157956,552187442,552186770,552159801,
       552154507,552187328,552187524,552157653,552157757,552158132,552158191,552157806,552187045,552187369,552189927,552187204,552160769,552159511,552184929,552189818,552190042,
       552155152,552187578,552190000,
]


In [ ]:
pre_aloc.dtypes

UserPS              int64
individual-id         str
Alias             float64
created_at            str
arquivo_origem        str
Origen_GPM            str
dtype: object

In [26]:
src_64 = pre_aloc.loc[pre_aloc['UserPS'].isin(src)]
src_64

,UserPS,individual-id,created_at,Origen_GPM,arquivo_origem
162,552185366,0552185366-01,06/02/2026 09:11,21 - Interior SP,pre_1000 Codigos Interior SP_060226_Josane.xlsx
312,552184929,0552184929-01,06/02/2026 09:11,21 - Interior SP,pre_1000 Codigos Interior SP_060226_Josane.xlsx
518,552185292,0552185292-01,06/02/2026 09:11,21 - Interior SP,pre_1000 Codigos Interior SP_060226_Josane.xlsx
1055,552154759,0552154759-01,29/12/2025 11:55,22 - Centro-Oeste,pre_1000 Preallocated Centro Oeste_291125.xlsx
1105,552154693,0552154693-01,29/12/2025 11:55,22 - Centro-Oeste,pre_1000 Preallocated Centro Oeste_291125.xlsx
...,...,...,...,...,...
8721,552187345,0552187345-01,06/02/2026 09:30,23 - Resto BR,pre_2000 Codigos Resto BR_060226_Josane.xlsx
8739,552187293,0552187293-01,06/02/2026 09:30,23 - Resto BR,pre_2000 Codigos Resto BR_060226_Josane.xlsx
8820,552187045,0552187045-01,06/02/2026 09:30,23 - Resto BR,pre_2000 Codigos Resto BR_060226_Josane.xlsx
8828,552187524,0552187524-01,06/02/2026 09:30,23 - Resto BR,pre_2000 Codigos Resto BR_060226_Josane.xlsx


# 3. Script Dados consolidados

Objetivo:
- Construir um consolidado de todos os ID's pre-alocados vs sua ultima transmissão;
- Se há uma ultima transmissão, é porque o ID já foi utilizado;

Etapas:
1. carregamento dos dados de interesse;
2. Inicializar a base de dados: ultima transmissão de todos os ids prealocados;
3. Criar base auxiliar apenas do periodo produtivo atual (2026-05-25 até 2026-06-23);
4. Desenvolver lógica de atualização da base consolidada;

## 3.1 Carregamento dos dados de interesse

In [ ]:
prefixo='do_cal'
# CARREGAMENTO

# CALENDARIO PRODUTIVO
df_cldar = functions.carregamento(
        
        path='C:/Users/luiz.farias/Numerator International/BKO - projeto-dados-ops/do_calendario_fiscal',
        prefixo='do_cal',
        fluxo_externo=True
        )    
# REP7
rep7 = functions.carregamento(
        path='C:/Users/luiz.farias/Numerator International/BKO - projeto-dados-ops/do_rep7/bruto',
        prefixo='Brasil-Reporte-7',
        fluxo_externo=True
    )
rep7 = functions.normalize(rep7)

rep_7_nf= rep7.copy()

# FILTRO PERIODO PRODUTIVO VS REP7
rep7, inicio, fim = functions.filtrar_periodo_vigente(
        rep7, 
        df_cldar,
        coluna_data='DiaDeCompra',
        data_referencia='2026-06-23'
    )

# TODOS OS IDS PREALOCADOS
pre_aloc = functions.carregamento(
        path='C:/Users/luiz.farias/Numerator International/BKO - projeto-dados-ops/do_preal_fornecedor/ref_prealoc',
        prefixo='pre',
        fluxo_externo=False

)

trocas = {
'{GroupId}' : 'UserPS',
'{IndividualId}' : 'individual-id',
'{CreationTimeStamp}' : 'created_at',
'Origen' : 'Origen_GPM'
}

pre_aloc.rename(columns=trocas, inplace=True)
pre_aloc.UserPS = pre_aloc.UserPS.astype(str)

/n
Entrada da função de normalização
Menor data encontrada: 01/04/2026  |  Maior data encontrada: 31/05/2026
/n
Dimensão entrada saida iguais/n
QTD dados original qmob: 155308
Coluna data: DiaDeCompra
Período encontrado: 2026-05-25 00:00:00 até 2026-06-23 00:00:00
QTD após filtro no periodo produtivo atual: 51066
/n/n
Entrada da Função de filtro de periodo
Menor data encontrada: 2026-03-24 00:00:00  |  Maior data encontrada: 2026-06-24 00:00:00
/n/n
Saida da Função de filtro de periodo : Dados filtrados
Menor data encontrada: 2026-05-25 00:00:00  |  Maior data encontrada: 2026-06-23 00:00:00
/n/n


## 3.2 Inicializar a base de dados: ultima transmissão de todos os ids prealocados;

Objetivo:
- Construir uma base consolidada de de informações por periodo produtivo para todos os ID's pre-alocados de Top Client:

Fluxo:
1. Identificar no relatório de atos Brutos | parametro: Rep7 Atualizado;
2. Identificar o periodo produtivo de cada trasmissão | parametro: Calendário fiscal da empresa;
3. Identificar os ids que tiveram transmissão | parametro: Pre_Aloc Atualizado;


In [ ]:
import pandas as pd

# Calculo encima da data máxima de transmissão de cada individuo
# Agrupamento de individuo por max date
max_transmission = (
    rep_7_nf
    .groupby('UserPS', as_index=False)
    .agg(DiaDeCompra=('DiaDeCompra', 'max'))
)

# Transformação de tipo de dados encima do calendário fiscal
df_cldar['Desde'] = pd.to_datetime(df_cldar['Desde'])
df_cldar['Hasta'] = pd.to_datetime(df_cldar['Hasta'])

# Cross join para identificar todas as combinações possivles de datas entre as bases
df_max_periodo = rep_7_nf.merge(df_cldar, how='cross')

# Filtro traz somente as compras que estão dentro de um periodo onde há o match entre a data máxima e desde e hasta
df_max_periodo = df_max_periodo.loc[
    (df_max_periodo['DiaDeCompra'] >= df_max_periodo['Desde']) &
    (df_max_periodo['DiaDeCompra'] <= df_max_periodo['Hasta'])
]

# Nessa base é possivel enteder todas as compras e o respectivo periodo produtivo
base_com_periodo = max_transmission.merge(
    df_max_periodo[
        ['UserPS', 'NumCompras','Desde', 'Hasta']
    ],
    on='UserPS',
    how='left'
) 

# Busca na base de dados somente as compras que estão dentro dos periodos produtivos aceitos
compras_periodo = (
    base_com_periodo
    .loc[
        (base_com_periodo['DiaDeCompra'] >= base_com_periodo['Desde']) &
        (base_com_periodo['DiaDeCompra'] <= base_com_periodo['Hasta'])
    ]
    
    # A agregação parte dos individuos e identicamos todas as compras do userPS que existem, elas ficam amarradas ao periodo produtivo;
    .groupby('UserPS', as_index=False)
    .agg(
        qtd_transmissions=('DiaDeCompra', 'count'),
        Desde=('Desde', 'first'),
        Hasta=('Hasta', 'first'),
        DiaDeCompra=('DiaDeCompra', 'first'),
        NumCompras=('NumCompras', 'sum')
    )
)

# O mês priduto é sempre identificado pela segunda fatia do periodo maior
# As conpras do periodo, agrupa transimssões, a ultima transmissão de cada periodo produtivo
compras_periodo['MesProdutivo'] = compras_periodo.Hasta.dt.strftime('%Y-%m')
base_com_periodo['MesProdutivo'] = base_com_periodo.Hasta.dt.strftime('%Y-%m')


# Agui, com base no periodo, identificamos somente a compra ultima compra do individuo
max_transmission = compras_periodo.groupby(['UserPS', 'qtd_transmissions','MesProdutivo','NumCompras']).agg(
    ultima_transmissao = ('DiaDeCompra','max')
).reset_index()

import pandas as pd

df_cldar['Desde'] = pd.to_datetime(df_cldar['Desde'])
df_cldar['Hasta'] = pd.to_datetime(df_cldar['Hasta'])

pd.set_option('display.max_colwidth', None)
out = pre_aloc.merge(max_transmission, on='UserPS', how='left')
out.UserPS = out.UserPS.astype(int)


path_temp_output= 'C:/Users/70089581/Numerator International/BKO - projeto-dados-ops/do_preal_fornecedor/ref_cosolidado/'

from datetime import datetime

data_atual = datetime.now().strftime("%Y%m%d")

# Definir os limites das faixas (bins) e os rótulos correspondentes
bins = [0, 4, 9, 14, 19, 24, float('inf')]
labels = ['1 a 4 atos', '5 a 9 atos', '10 a 14 atos', '15 a 19 atos', '20 a 24 atos', '25+ atos']

# Criar a nova coluna 'FaixaNumCompras' usando pd.cut
out['FaixaNumCompras'] = pd.cut(out['NumCompras'], bins=bins, labels=labels, right=True)

out.to_excel(path_temp_output + f'consolidado_ids_tc_rep7_{data_atual}.xlsx', index = False)


In [ ]:

def inf_consolidada(rep_7_nf, df_cldar, pre_aloc):
    import pandas as pd
    from datetime import datetime


    # =========================
    # 1. Preparação
    # =========================

    rep_7_nf['DiaDeCompra'] = pd.to_datetime(rep_7_nf['DiaDeCompra'])
    df_cldar['Desde'] = pd.to_datetime(df_cldar['Desde'])
    df_cldar['Hasta'] = pd.to_datetime(df_cldar['Hasta'])

    pre_aloc['UserPS'] = pre_aloc['UserPS'].astype(str)
    rep_7_nf['UserPS'] = rep_7_nf['UserPS'].astype(str)

    df_periodos = df_cldar.copy()
    df_periodos['MesProdutivo'] = df_periodos['Hasta'].dt.strftime('%Y-%m')

    # =========================
    # 2. Base ID x Período
    # =========================

    ids_periodos = pre_aloc[['UserPS']].drop_duplicates().merge(
        df_periodos[['MesProdutivo', 'Desde', 'Hasta']],
        how='cross'
    )

    # =========================
    # 3. Transmissões com período
    # =========================

    base_com_periodo = rep_7_nf.merge(
        df_periodos[['MesProdutivo', 'Desde', 'Hasta']],
        how='cross'
    )

    base_com_periodo = base_com_periodo.loc[
        (base_com_periodo['DiaDeCompra'] >= base_com_periodo['Desde']) &
        (base_com_periodo['DiaDeCompra'] <= base_com_periodo['Hasta'])
    ].copy()

    # =========================
    # 4. Agregado por ID e período
    # =========================

    transmissao_por_periodo = (
        base_com_periodo
        .groupby(['UserPS', 'MesProdutivo'], as_index=False)
        .agg(
            qtd_transmissions=('DiaDeCompra', 'count'),
            primeira_transmissao=('DiaDeCompra', 'min'),
            ultima_transmissao=('DiaDeCompra', 'max'),
            NumCompras=('NumCompras', 'sum')
        )
    )

    # =========================
    # 5. Enriquecer ID x período
    # =========================

    out = ids_periodos.merge(
        transmissao_por_periodo,
        on=['UserPS', 'MesProdutivo'],
        how='left'
    )

    # =========================
    # 6. Trazer colunas originais do pre_aloc
    # =========================

    out = out.merge(
        pre_aloc,
        on='UserPS',
        how='left'
    )

    # =========================
    # 7. Faixa de compras
    # =========================

    bins = [0, 4, 9, 14, 19, 24, float('inf')]
    labels = [
        '1 a 4 atos',
        '5 a 9 atos',
        '10 a 14 atos',
        '15 a 19 atos',
        '20 a 24 atos',
        '25+ atos'
    ]

    out['FaixaNumCompras'] = pd.cut(
        out['NumCompras'],
        bins=bins,
        labels=labels,
        right=True
    )

    return out 

## 3.2 Script de atualização de dados;
Objetivo: 
- Atualizar uma base histórica fixa para os dados da top client:

Etapas:
1. Entrar com a path de saída | Parametro: caminho fisico dos dados;
2. Atualizar arquivo e manter somente as atualizaçõe | Parâmetro: Arquivo com novos dados;

In [ ]:
def atualizar_consolidado(cons):
    import pandas as pd
    from pathlib import Path
    from datetime import datetime
    path = Path('C:/Users/70089581/Numerator International/BKO - projeto-dados-ops/do_preal_fornecedor/ref_cosolidado')

    arquivo_historico = path / 'consolidado_ids_tc_rep7_historico.xlsx' 

    # cons_nw é o out que você acabou de gerar no script atual
    cons_nw = cons.copy()

    cons_nw['data_atualizacao'] = datetime.now()

    chave = ['UserPS', 'MesProdutivo']

    '''if arquivo_historico.exists():
        print('Entrou no if')
        historico = pd.read_excel(arquivo_historico)

        historico['data_atualizacao'] = pd.to_datetime(historico['data_atualizacao'])
        cons_nw['data_atualizacao'] = pd.to_datetime(cons_nw['data_atualizacao'])

        base_atualizada = pd.concat(
            [historico, cons_nw],
            ignore_index=True
        )

        base_atualizada = (
            base_atualizada
            .sort_values('data_atualizacao')
            .drop_duplicates(subset=chave, keep='last')
        )
        base_atualizada.to_excel(arquivo_historico,index=False)'''


    
    print('entrou no else')
    base_atualizada = cons_nw.copy()

    base_atualizada.to_excel(
        arquivo_historico,
        index=False
    )



In [ ]:
out = functions.inf_consolidada(rep_7_nf,df_cldar,pre_aloc)

atualizar_consolidado(out)

NameError: name 'rep_7_nf' is not defined

In [ ]:


def atualizar_consolidado(path,cons):
    import pandas as pd
    from pathlib import Path
    from datetime import datetime
    path = Path(
        'C:/Users/70089581/Numerator International/BKO - projeto-dados-ops/do_preal_fornecedor/ref_cosolidado/'
    )

    arquivo_historico = path / 'consolidado_ids_tc_rep7_historico.xlsx'

    # cons_nw é o out que você acabou de gerar no script atual
    cons_nw = cons.copy()

    cons_nw['data_atualizacao'] = datetime.now()

    chave = ['UserPS', 'MesProdutivo']

    if arquivo_historico.exists():
        historico = pd.read_excel(arquivo_historico)

        historico['data_atualizacao'] = pd.to_datetime(historico['data_atualizacao'])
        cons_nw['data_atualizacao'] = pd.to_datetime(cons_nw['data_atualizacao'])

        base_atualizada = pd.concat(
            [historico, cons_nw],
            ignore_index=True
        )

        base_atualizada = (
            base_atualizada
            .sort_values('data_atualizacao')
            .drop_duplicates(subset=chave, keep='last')
        )

    else:
        base_atualizada = cons_nw.copy()

        base_atualizada.to_excel(
        arquivo_historico,
        index=False
    )

### 3.3 normlaização de dados

In [29]:
#normalizacção de origem
import pandas as pd

ls = pre_aloc.groupby(['arquivo_origem','Origen_GPM'])['UserPS'].nunique().reset_index()
PATH = 'C:/Users/luiz.farias/Numerator International/BKO - projeto-dados-ops/do_preal_fornecedor/ref_prealoc/'
for nm_arq in ls.arquivo_origem:
    temp = pd.read_excel(PATH + nm_arq)

    temp.Origen = temp.Origen.replace(22,'22 - Centro-Oeste')
    temp.Origen = temp.Origen.replace('22 - Centro Oeste','22 - Centro-Oeste')
    temp.Origen = temp.Origen.replace(21,'21 - Interior SP')
    temp.Origen = temp.Origen.replace(23,'23 - Resto BR')
    temp.Origen = temp.Origen.replace(20,'20 - Nordeste')
    temp.Origen = temp.Origen.replace(4,'4 - IHS')
    temp.Origen = temp.Origen.replace('IHS','4 - IHS')

    temp.to_excel(PATH + nm_arq, index=False)
    



In [ ]:
df_criterio = functions.carregamento(
    path='C:/Users/70089581/Numerator International/BKO - projeto-dados-ops/do_base_criterio',
    prefixo='do_base',
    fluxo_externo=False
)

c:\Users\70089581\OneDrive - Kantar\Área de Trabalho\Workspace\Projeto_InOut_Top_Client\functions.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(dfs, ignore_index=True)
